In [1]:
from pathlib import Path

BASE = Path(r"E:\AI_Detect")

folders = [

    "data/image/human/raw/wikimedia",
    "data/image/human/raw/nasa",
    "data/image/human/raw/loc",
    "data/image/human/raw/archive",

    "data/image/human/metadata"
]

for f in folders:

    (BASE / f).mkdir(
        parents=True,
        exist_ok=True
    )

print("Folders created.")

Folders created.


In [3]:
WIKIMEDIA_TARGET = 1000
NASA_TARGET = 1000
LOC_TARGET = 1000
ARCHIVE_TARGET = 1000
WIKIMEDIA_TARGET = 5000

In [4]:
import pandas as pd
import requests

from pathlib import Path
from tqdm import tqdm

BASE = Path(r"E:\AI_Detect")

META_FILE = (
    BASE /
    "data/image/human/metadata/image_metadata.csv"
)

if META_FILE.exists():

    meta_df = pd.read_csv(META_FILE)

else:

    meta_df = pd.DataFrame()

existing_files = set(
    meta_df.filename.tolist()
) if len(meta_df) else set()

In [ ]:
NASA_TARGET = 1000

save_dir = (
    BASE /
    "data/image/human/raw/nasa"
)

saved = 0

page = 1

pbar = tqdm(total=NASA_TARGET)

while saved < NASA_TARGET:

    url = (
        "https://images-api.nasa.gov/search"
    )

    params = {

        "media_type": "image",

        "page": page

    }

    try:

        r = requests.get(
            url,
            params=params,
            timeout=30
        )

        items = (
            r.json()
            ["collection"]
            ["items"]
        )

        if len(items) == 0:
            break

        for item in items:

            try:

                img_url = (
                    item["links"][0]["href"]
                )

                fname = img_url.split("/")[-1]

                if fname in existing_files:
                    continue

                img = requests.get(
                    img_url,
                    timeout=30
                )

                with open(
                    save_dir / fname,
                    "wb"
                ) as f:

                    f.write(img.content)

                existing_files.add(fname)

                meta_df = pd.concat([

                    meta_df,

                    pd.DataFrame([{

                        "filename": fname,

                        "source": "nasa",

                        "url": img_url

                    }])

                ])

                saved += 1

                pbar.update(1)

                if saved >= NASA_TARGET:
                    break

            except:
                pass

        page += 1

    except:
        break

pbar.close()


from pathlib import Path
from tqdm import tqdm
import pandas as pd
import requests
import os

# ====================================================
# PATHS
# ====================================================

BASE = Path(r"E:\AI_Detect")

RAW_DIR = BASE / "data/image/human/raw"

META_DIR = BASE / "data/image/human/metadata"

RAW_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

META_FILE = META_DIR / "wikimedia_metadata.csv"

# ====================================================
# CONFIG
# ====================================================

TARGET_IMAGES = 5000

YEAR_CUTOFF = 2021

# ====================================================
# EXISTING FILES
# ====================================================

existing = set()

for fp in RAW_DIR.glob("*"):

    existing.add(fp.name)

print("Existing images:", len(existing))

# ====================================================
# LOAD OLD METADATA
# ====================================================

metadata_rows = []

if META_FILE.exists():

    try:

        old = pd.read_csv(META_FILE)

        metadata_rows = old.to_dict("records")

    except:
        pass

# ====================================================
# WIKIMEDIA API
# ====================================================

saved = 0

session = requests.Session()

pbar = tqdm(total=TARGET_IMAGES)

offset = 0

while saved < TARGET_IMAGES:

    params = {

        "action": "query",

        "format": "json",

        "generator": "categorymembers",

        "gcmtitle": "Category:Featured_pictures_on_Wikimedia_Commons",

        "gcmlimit": 50,

        "gcmtype": "file",

        "prop": "imageinfo",

        "iiprop": "url|timestamp",

    }

    try:

        r = session.get(

            "https://commons.wikimedia.org/w/api.php",

            params=params,

            timeout=30

        )

        data = r.json()

        pages = data.get("query", {}).get("pages", {})

        if len(pages) == 0:
            break

        for page_id in pages:

            page = pages[page_id]

            title = page["title"]

            info = page["imageinfo"][0]

            image_url = info["url"]

            timestamp = info["timestamp"]

            year = int(timestamp[:4])

            if year > YEAR_CUTOFF:
                continue

            ext = image_url.split(".")[-1].lower()

            fname = title.replace("File:", "")

            fname = fname.replace("/", "_")

            if fname in existing:
                continue

            try:

                img = requests.get(
                    image_url,
                    timeout=30
                )

                save_path = RAW_DIR / fname

                with open(
                    save_path,
                    "wb"
                ) as f:

                    f.write(img.content)

                existing.add(fname)

                metadata_rows.append({

                    "filename": fname,

                    "url": image_url,

                    "year": year,

                    "source": "wikimedia"

                })

                saved += 1

                pbar.update(1)

                if saved >= TARGET_IMAGES:
                    break

            except:
                continue

    except Exception as e:

        print(e)

        continue

pbar.close()

# ====================================================
# SAVE METADATA
# ====================================================

meta = pd.DataFrame(metadata_rows)

meta.to_csv(
    META_FILE,
    index=False
)

print()
print("Downloaded:", saved)
print("Metadata:", META_FILE)


save_dir = (
    BASE /
    "data/image/human/raw/loc"
)

saved = 0

offset = 0

pbar = tqdm(total=LOC_TARGET)

while saved < LOC_TARGET:

    url = (
        "https://www.loc.gov/photos/"
    )

    params = {

        "fo": "json",

        "c": 100,

        "sp": offset

    }

    try:

        r = requests.get(
            url,
            params=params,
            timeout=30
        )

        data = r.json()

        results = data.get(
            "results",
            []
        )

        if len(results) == 0:
            break

        for item in results:

            try:

                img_url = item["image_url"][0]

                fname = img_url.split("/")[-1]

                if fname in existing_files:
                    continue

                img = requests.get(
                    img_url,
                    timeout=30
                )

                with open(
                    save_dir / fname,
                    "wb"
                ) as f:

                    f.write(img.content)

                existing_files.add(fname)

                meta_df = pd.concat([

                    meta_df,

                    pd.DataFrame([{

                        "filename": fname,

                        "source": "loc",

                        "url": img_url

                    }])

                ])

                saved += 1

                pbar.update(1)

                if saved >= LOC_TARGET:
                    break

            except:
                pass

        offset += 1

    except:
        break

pbar.close()

save_dir = (
    BASE /
    "data/image/human/raw/archive"
)

saved = 0

page = 1

pbar = tqdm(total=ARCHIVE_TARGET)

while saved < ARCHIVE_TARGET:

    params = {

        "q": (
            "mediatype:image "
            "AND year:[2000 TO 2021]"
        ),

        "rows": 100,

        "page": page,

        "output": "json"

    }

    try:

        r = requests.get(

            "https://archive.org/advancedsearch.php",

            params=params,

            timeout=30

        )

        docs = (
            r.json()
            ["response"]
            ["docs"]
        )

        if len(docs) == 0:
            break

        for doc in docs:

            try:

                identifier = doc["identifier"]

                meta_url = (
                    f"https://archive.org/metadata/"
                    f"{identifier}"
                )

                meta = requests.get(
                    meta_url
                ).json()

                files = meta.get(
                    "files",
                    []
                )

                for f in files:

                    name = f["name"]

                    if not (
                        name.endswith(".jpg")
                        or
                        name.endswith(".jpeg")
                        or
                        name.endswith(".png")
                    ):
                        continue

                    url = (
                        f"https://archive.org/download/"
                        f"{identifier}/{name}"
                    )

                    fname = (
                        identifier +
                        "_" +
                        name
                    )

                    if fname in existing_files:
                        continue

                    img = requests.get(
                        url,
                        timeout=30
                    )

                    with open(
                        save_dir / fname,
                        "wb"
                    ) as out:

                        out.write(
                            img.content
                        )

                    existing_files.add(
                        fname
                    )

                    meta_df = pd.concat([

                        meta_df,

                        pd.DataFrame([{

                            "filename": fname,

                            "source": "archive",

                            "url": url

                        }])

                    ])

                    saved += 1

                    pbar.update(1)

                    if saved >= ARCHIVE_TARGET:
                        break

            except:
                pass

        page += 1

    except:
        break

pbar.close()

meta_df.to_csv(
    META_FILE,
    index=False
)

print()
print("Total Images:",
      len(meta_df))

print("Metadata Saved:")
print(META_FILE)

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [05:48<00:00,  2.87it/s]


Existing images: 4


  0%|                                                                                         | 0/5000 [00:00<?, ?it/s]

Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (